BERT

In [37]:
import pandas as pd
import numpy as np
import tf_keras as keras
import tensorflow as tf
from transformers import BertTokenizer,TFBertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report
from sklearn.preprocessing import OneHotEncoder

In [20]:
df = pd.read_csv(r"/content/sample_data/IMDB Dataset.csv")

print(df.head(5))

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [30]:
oe = OneHotEncoder(drop="first",sparse_output=False)
y = oe.fit_transform(df[["sentiment"]])

X = df["review"]

X_train_val,X_test,y_train_val,y_test = train_test_split(
    X.tolist(),y.tolist(),train_size=5000,test_size=1000,random_state=42,stratify=y
)

X_train,X_val,y_train,y_val = train_test_split(
    X_train_val,y_train_val,test_size=500,random_state=42,stratify=y_train_val
)

In [31]:
Tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [33]:
train_encodings = dict(Tokenizer(X_train,padding=True,truncation=True,max_length=128,return_tensors="tf"))
val_encodings = dict(Tokenizer(X_val,padding=True,truncation=True,max_length=128,return_tensors="tf"))
test_encodings = dict(Tokenizer(X_test,padding=True,truncation=True,max_length=128,return_tensors="tf"))
train_label = tf.convert_to_tensor(y_train)
val_label = tf.convert_to_tensor(y_val)
test_label = tf.convert_to_tensor(y_test)

In [38]:
Model = TFBertForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=2)
optimizer = keras.optimizers.Adam(learning_rate=2e-5)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [40]:
Model.compile(optimizer=optimizer,loss=loss,metrics=["accuracy"])
history = Model.fit(train_encodings,train_label,validation_data=(val_encodings,val_label),epochs=10)

Epoch 1/10
141/141 [==============================] - 200s 890ms/step - loss: 0.4334 - accuracy: 0.7882 - val_loss: 0.3383 - val_accuracy: 0.8540
Epoch 2/10
141/141 [==============================] - 123s 869ms/step - loss: 0.2290 - accuracy: 0.9078 - val_loss: 0.3299 - val_accuracy: 0.8640
Epoch 3/10
141/141 [==============================] - 125s 889ms/step - loss: 0.1238 - accuracy: 0.9564 - val_loss: 0.3926 - val_accuracy: 0.8560
Epoch 4/10
141/141 [==============================] - 128s 905ms/step - loss: 0.0693 - accuracy: 0.9796 - val_loss: 0.3982 - val_accuracy: 0.8620
Epoch 5/10
141/141 [==============================] - 128s 909ms/step - loss: 0.0335 - accuracy: 0.9920 - val_loss: 0.5203 - val_accuracy: 0.8560
Epoch 6/10
141/141 [==============================] - 128s 905ms/step - loss: 0.0411 - accuracy: 0.9873 - val_loss: 0.4531 - val_accuracy: 0.8500
Epoch 7/10
141/141 [==============================] - 127s 904ms/step - loss: 0.0252 - accuracy: 0.9916 - val_loss: 0.5074 -

In [41]:
pred = Model.predict(test_encodings)
pred = pred.logits
probabilities = tf.nn.softmax(pred,axis=1)
predictions = tf.argmax(probabilities,axis=1)

32/32 [==============================] - 12s 286ms/step


In [42]:
accuracy = accuracy_score(predictions,test_label)
print("Accuracy score: ",accuracy)
report = classification_report(predictions,test_label)
print("Classification report: ")
print(report)

Accuracy score:  0.87
Classification report: 
              precision    recall  f1-score   support

           0       0.84      0.89      0.87       470
           1       0.90      0.85      0.87       530

    accuracy                           0.87      1000
   macro avg       0.87      0.87      0.87      1000
weighted avg       0.87      0.87      0.87      1000

